# Fine-tuning Vision Transformers with vit-trainer

This notebook demonstrates how to use the `vit-trainer` package to fine-tune Vision Transformers for image classification.

**Author**: John Hodge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/PyTorch-Vision-Transformers-ViT/blob/main/notebooks/tutorial.ipynb)

## Installation

Install the vit-trainer package and its dependencies:

In [ ]:
# Install vit-trainer.
#
# This installs the fix branch so the notebook runs the corrected attention,
# reconciled CLI/config, run manifests, and verified export. The onnxscript
# pin lets the export cell work on torch >= 2.9 (current Colab).
#
# Swap to `!pip install "vit-trainer[export]"` once these land in a release.
!pip install "vit-trainer[export] @ git+https://github.com/jman4162/PyTorch-Vision-Transformers-ViT.git@fix/attention-cli-provenance"

## Quick Start

The simplest way to train a Vision Transformer:

In [ ]:
from vit_trainer import (
    Trainer,
    load_model,
    get_cifar10_loaders,
    CIFAR10_CLASSES,
)

# Load data
train_loader, val_loader, test_loader = get_cifar10_loaders(
    batch_size=64,
    seed=42,
)

print(f"Train: {len(train_loader.dataset)} samples")
print(f"Validation: {len(val_loader.dataset)} samples")
print(f"Test: {len(test_loader.dataset)} samples")

In [ ]:
# Load pretrained ViT model
model = load_model("vit_b_16", num_classes=10)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create trainer with modern techniques.
# `metadata` is written into every checkpoint, so the weights carry the
# config that produced them.
from vit_trainer import TrainingConfig

config = TrainingConfig(batch_size=64, epochs=10, seed=42)

trainer = Trainer(
    model=model,
    lr=config.lr,
    weight_decay=config.weight_decay,
    warmup_epochs=config.warmup_epochs,
    use_amp=config.use_amp,  # Mixed precision; effective on Tensor Core GPUs
    gradient_clip=config.gradient_clip,
    metadata={"config": config.to_dict()},
)

# Train the model
history = trainer.fit(
    train_loader,
    val_loader,
    epochs=config.epochs,
    patience=config.patience,
)

In [ ]:
# Evaluate on test set
loss, accuracy = trainer.evaluate(test_loader)
print(f"\nTest Accuracy: {accuracy:.2f}%")

## Recording What Produced That Number

An accuracy on its own is not reproducible. The commit, environment, GPU, seed,
and data split all have to travel with it, or the number is an anecdote.

In [ ]:
import json

from vit_trainer import collect_run_metadata, save_run_metadata
from vit_trainer.data import make_split_indices
from vit_trainer.provenance import hash_indices

# Same call the loader makes, so this identifies the split actually used.
_, val_indices = make_split_indices(50000, train_split=config.train_split, seed=config.seed)

manifest = collect_run_metadata(
    config,
    split_hash=hash_indices(val_indices),
    results={
        "test_accuracy": accuracy,
        "test_loss": loss,
        "best_epoch": int(min(range(len(history["val_loss"])), key=lambda i: history["val_loss"][i])) + 1,
        "epochs_run": len(history["val_loss"]),
        "mean_epoch_seconds": sum(history["epoch_time"]) / len(history["epoch_time"]),
        "peak_gpu_memory_mb": trainer.peak_memory_mb(),
    },
)

save_run_metadata(manifest, "run.json")
print(json.dumps(manifest, indent=2)[:1200])

## Visualizing Training Progress

In [ ]:
from vit_trainer.evaluation import plot_training_history

plot_training_history(history)

## Detailed Evaluation

In [ ]:
from vit_trainer import (
    get_predictions,
    compute_metrics,
    plot_confusion_matrix,
)

# Get all predictions
y_pred, y_true, probs = get_predictions(model, test_loader)

# Compute detailed metrics
metrics = compute_metrics(y_true, y_pred, CIFAR10_CLASSES)
print(metrics["classification_report"])

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(y_true, y_pred, CIFAR10_CLASSES)

## Attention Visualization

Attention maps show which patches the final-layer CLS token pooled from. That is
useful for spotting a model keying on background or borders, but it is a
diagnostic, not an explanation of the prediction — see
[Attention is not Explanation](https://arxiv.org/abs/1902.10186).

Two things have to be right for the map to mean anything:

1. The captured attention must come from the *actual* forward pass. Skip the
   positional embedding and you still get a smooth, plausible-looking heatmap
   from a representation the model never sees.
2. The map must be normalized to [0, 1] before any 8-bit conversion. A CLS
   attention row is a distribution over 196 patches, averaging about 0.005, so
   quantizing first flattens most of it to zero.

`forward_with_attention` returns logits alongside the attention so the first
condition can be asserted rather than assumed.

In [ ]:
import torch

from vit_trainer import forward_with_attention, visualize_samples_with_attention

# Check that the attention we are about to plot comes from the same forward
# pass the model uses to predict.
images, _ = next(iter(test_loader))
device = next(model.parameters()).device
with torch.no_grad():
    _, manual_logits = forward_with_attention(model, images[:2].to(device))
    assert torch.allclose(manual_logits, model(images[:2].to(device)), atol=1e-4)
print("Attention path reproduces the model's logits.")

# Visualize attention on test samples (seeded so the figure is reproducible)
visualize_samples_with_attention(
    model,
    test_loader.dataset,
    CIFAR10_CLASSES,
    num_samples=4,
    seed=0,
)

## Single Image Prediction

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
from vit_trainer import get_val_transform, visualize_attention, show_attention_on_image

def predict_single_image(model, image_path, device=None):
    """Predict class for a single image with attention visualization."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load and transform image
    image = Image.open(image_path).convert("RGB")
    transform = get_val_transform()
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    model.eval()
    model.to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        pred_idx = outputs.argmax(dim=1).item()

    # Get attention
    attn_map = visualize_attention(model, input_tensor[0], device=device)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(attn_map, cmap="hot")
    axes[1].set_title("CLS attention (last layer)")
    axes[1].axis("off")

    overlay = show_attention_on_image(image.resize((224, 224)), attn_map)
    axes[2].imshow(overlay)

    pred_label = CIFAR10_CLASSES[pred_idx]
    confidence = probs[pred_idx].item()
    axes[2].set_title(f"{pred_label}: {confidence:.1%}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    # Print top-5 predictions
    print("\nTop-5 Predictions:")
    top_probs, top_indices = torch.topk(probs, 5)
    for prob, idx in zip(top_probs, top_indices):
        print(f"  {CIFAR10_CLASSES[idx]}: {prob.item():.2%}")

# Example usage (uncomment and provide your own image):
# predict_single_image(model, "path/to/your/image.jpg")

## Export, and Checking the Export

Serializing a model and deploying one are different claims. The first is about
producing a file; the second is about that file computing what PyTorch computed.

In [ ]:
from vit_trainer import ExportConfig

export_config = ExportConfig(output_path="vit_cifar10.onnx", opset_version=14)
export_config.export(model)
print("Model exported to vit_cifar10.onnx")

# `onnx.checker` only says the graph is well formed. Whether ONNX Runtime
# computes the same numbers as PyTorch is a separate question, so ask it:
# identical inputs, compare logits and predicted classes at several batch sizes.
results = export_config.verify(model, batch_sizes=(1, 4))

for batch_size, stats in results["batches"].items():
    print(
        f"  batch {batch_size}: max logit delta "
        f"{stats['max_logit_delta']:.2e}, predictions match"
    )
print(f"Model size: {results['size_mb']:.1f} MB")

## Using Configuration Files

In [ ]:
## CLI Usage

The package also provides a command-line interface. `train` writes a run
manifest next to the checkpoint, and `export` verifies the exported file against
PyTorch before reporting success.

```bash
# Train a model (writes models/best_model_vit_b_16_cifar10.run.json)
vit-train train --model vit_b_16 --dataset cifar10 --epochs 10

# Flags you type override the YAML; everything else comes from the file
vit-train train --config configs/default.yaml --batch-size 32 --no-amp

# Evaluate — variant and dataset are read from the checkpoint
vit-train eval --checkpoint models/best_model_vit_b_16_cifar10.pt

# Predict on a single image
vit-train predict --checkpoint best_model.pt --image cat.jpg --show-attention

# Export to ONNX and check the runtime reproduces PyTorch's logits
vit-train export --checkpoint best_model.pt --output model.onnx
```

## CLI Usage

The package also provides a command-line interface:

```bash
# Train a model
vit-train train --model vit_b_16 --dataset cifar10 --epochs 10

# Evaluate a trained model
vit-train eval --checkpoint best_model.pt --dataset cifar10

# Predict on a single image
vit-train predict --checkpoint best_model.pt --image cat.jpg

# Export to ONNX
vit-train export --checkpoint best_model.pt --output model.onnx
```

## Conclusion

This tutorial covered:

1. **Loading data**: `get_cifar10_loaders()` with a seeded, deterministic split
   and validation transforms that skip augmentation
2. **Loading models**: `load_model()` with pretrained weights
3. **Training**: `Trainer` with AMP, warmup, gradient clipping, early stopping
4. **Provenance**: a run manifest recording commit, environment, GPU, seed, and
   split hash, so the accuracy above means something to someone else
5. **Evaluation**: metrics, confusion matrices, classification reports
6. **Attention**: maps captured from the real forward pass, read as a diagnostic
7. **Export**: ONNX, verified against PyTorch's logits rather than assumed

What this package does not do: resume interrupted runs (checkpoints omit
optimizer state), train on datasets other than CIFAR-10/100, or run distributed.

For the original pre-package tutorial, see
`Fine_tuning_Vision_Transformers_ViT_with_PyTorch.ipynb`. Note that the 97.65%
figure quoted in the README came from that notebook, not from this package.